# Step 3.3 review: are the prices right, and which tickers changed between the trades file and the price file?

The returns in `2) ComputeReturns/returns_table.csv` are only as good as `1) FetchPrices/prices.csv`, and the price file was fetched in two rounds while the ticker review (section 5 of TradesCleaning.ipynb) was still moving tickers. This notebook answers two questions for the authors before the analysis starts:

1. **Reconciliation.** Which tickers of the final trade file (`2) Trades/Trump_trades_2025_2026.xlsx`, sheet `Trades`) have a price series, which were changed or dropped on the way, and why does the price file hold symbols the trades file no longer has.
2. **Price quality.** Whether any series looks wrong: duplicates, holes, stale prints after a delisting, series that start or end at odd dates, daily moves that are too large to be true, breaks at a symbol change, and dividend adjustments that are missing.

Nothing is changed here. The findings go to `PricesReview.xlsx`, and the two or three things that need a decision are listed at the end. The trade workbook (`2) Trades/Trump_trades_2025_2026.xlsx`, sheet `Trades`) is read only.

In [1]:
import re, json, sqlite3
from pathlib import Path
import numpy as np, pandas as pd
import requests
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 200)

TRADES   = Path("../../2) Trades/Trump_trades_2025_2026.xlsx")       # read only; sheet Trades = the final file
REVIEW   = Path("../../2) Trades/TickerReview.xlsx")                 # the review sheets of section 5
PURCH    = Path("../RawData/purchases.csv")
PRICES   = Path("../1) FetchPrices/prices.csv"); FACTORS = Path("../2) ComputeReturns/factors_daily.csv")
BAD_REV  = Path("PricesReview.xlsx")                               # sheet ExtremeMoves: Filip's extreme-move review, read by ComputeReturns.py
COMPANY_DB = Path("../../4) CompanyDB/company.db")
OUT_XLSX = Path("PricesReview.xlsx")
SAMPLE_START, SAMPLE_END, EVENT_DAYS = pd.Timestamp("2025-01-21"), pd.Timestamp("2026-08-31"), 10

trades = pd.read_excel(TRADES, sheet_name="Trades"); trades["Traded"] = pd.to_datetime(trades["Traded"])
purch = pd.read_csv(PURCH, parse_dates=["trade_date"]); left_out = pd.read_excel(TRADES, sheet_name="PurchasesLeftOut", parse_dates=["trade_date"])
prices = pd.read_csv(PRICES, parse_dates=["date"])
W = prices.pivot(index="date", columns="ticker", values="adj_close").sort_index()
cal = pd.DatetimeIndex(pd.read_csv(FACTORS, parse_dates=[0]).iloc[:, 0])          # trading days of the sample (French calendar)
ent = pd.read_sql("select * from entities", sqlite3.connect(COMPANY_DB))
wc = pd.read_excel(REVIEW, sheet_name="WrongCompanyAfterPrices"); rn = pd.read_excel(REVIEW, sheet_name="TickerRenames"); ne = pd.read_excel(REVIEW, sheet_name="NonEquity")
bad_rev = pd.read_excel(BAD_REV, sheet_name="ExtremeMoves"); bad_rev["Date"] = pd.to_datetime(bad_rev["Date"]); bad_rev["Decision"] = bad_rev["Decision"].astype(str).str.strip()

T_all = set(trades["Ticker"]); T_buy = set(trades.loc[trades["Transaction"].str.lower().str.startswith("purchase"), "Ticker"])
P = sorted(purch["ticker"].unique()); PR = set(W.columns)
print(f"trades: {len(trades)} rows, {len(T_all)} tickers ({len(T_buy)} purchased, {len(T_all - T_buy)} sale-only)")
print(f"step-5 purchases: {len(purch)} rows, {len(P)} tickers | left out: {len(left_out)} purchases, {left_out['ticker'].nunique()} tickers")
print(f"prices.csv: {len(prices)} rows, {len(PR)} symbols, {W.index.min().date()} .. {W.index.max().date()} | sample calendar: {len(cal)} trading days")

trades: 27628 rows, 1620 tickers (1599 purchased, 21 sale-only)
step-5 purchases: 18797 rows, 1594 tickers | left out: 21 purchases, 5 tickers
prices.csv: 651767 rows, 1639 symbols, 2025-01-10 .. 2026-08-31 | sample calendar: 405 trading days


## 1) Reconciliation: trades file against price file

The trades file has one ticker per company after Step 2.5 (latest symbol). The step-5 purchase table is the purchased subset of it minus five tickers FMP cannot price (reused symbols, re-registered companies). The price file is older than the final Step 2.5 round, so it also holds the symbols that Step 2.5 moved away from; those series are simply never read, because step 5 keeps only the tickers of the purchase table. Each superseded symbol is classified below from the review sheets of section 5.

In [2]:
apply = rn[rn["Decision"].astype(str).str.strip() == "apply"]
def classify(s):
    if s in set(wc["Ticker"]):
        return "wrong company (section 5)", "disclosed company moved to " + ", ".join(sorted(set(wc.loc[wc["Ticker"] == s, "CorrectedTicker"])))
    if s in set(apply["OldTicker"]):
        return "renamed (section 5)", "all rows moved to " + apply.loc[apply["OldTicker"] == s, "LatestTicker"].iloc[0]
    if s in set(rn["OldTicker"]):
        d = rn.loc[rn["OldTicker"] == s, "Decision"].iloc[0]; return f"rename {d}", "old ticker kept in the trades"
    if s in set(ne["Ticker"]):
        return "non-equity (section 5)", "rows dropped from the trades: " + "; ".join(ne.loc[ne["Ticker"] == s, "Company"].astype(str))
    if s in set(rn["LatestTicker"]):
        old = rn.loc[rn["LatestTicker"] == s].iloc[0]; return f"rename target, decision {old['Decision']}", f"fetched when {old['OldTicker']} -> {s} was still applied; the trades keep {old['OldTicker']}"
    return "unclassified", ""
superseded = pd.DataFrame([(s, *classify(s), int(W[s].notna().sum()), W[s].first_valid_index().date(), W[s].last_valid_index().date()) for s in sorted(PR - T_all)],
                          columns=["symbol", "why not in the trades", "detail", "price_days", "first", "last"])
sale_only = sorted(T_all - T_buy)
universe = pd.DataFrame([
    ("trades file: distinct tickers", len(T_all), ""),
    ("  of which purchased at least once", len(T_buy), ""),
    ("  of which sale-only (no price needed)", len(sale_only), ", ".join(sale_only)),
    ("step-5 purchase table: tickers", len(P), ""),
    ("  purchased tickers left out (FMP cannot price them)", len(set(left_out['ticker'])), ", ".join(sorted(set(left_out['ticker'])))),
    ("prices.csv: symbols", len(PR), ""),
    ("  used by step 5 (= purchase table)", len(PR & set(P)), ""),
    ("  purchased tickers without any series", len(set(P) - PR), ", ".join(sorted(set(P) - PR)) or "none"),
    ("  superseded symbols, never read by step 5", len(PR - T_all), ""),
    ("  left-out tickers whose symbol has a series (another issuer or a late start)", len(PR & set(left_out['ticker'])), ", ".join(sorted(PR & set(left_out['ticker'])))),
], columns=["item", "count", "tickers"])
display(universe)
print("superseded symbols by category:"); display(superseded["why not in the trades"].value_counts().rename_axis("category").reset_index(name="symbols"))
display(superseded)
assert not (set(P) - PR), "a purchased ticker has no price series"

,item,count,tickers
0,trades file: distinct tickers,1620,
1,of which purchased at least once,1599,
2,of which sale-only (no price needed),21,"BILL, CDE, CELH, CHX, CLVT, CNXC, COLD, FBIN, ..."
3,step-5 purchase table: tickers,1594,
4,purchased tickers left out (FMP cannot price...,5,"BRKL, NVRI, PINC, RNAM, WOLF"
5,prices.csv: symbols,1639,
6,used by step 5 (= purchase table),1594,
7,purchased tickers without any series,0,none
8,"superseded symbols, never read by step 5",40,
9,left-out tickers whose symbol has a series (...,5,"BRKL, NVRI, PINC, RNAM, WOLF"


superseded symbols by category:


,category,symbols
0,renamed (section 5),21
1,wrong company (section 5),15
2,non-equity (section 5),3
3,"rename target, decision skip",1


,symbol,why not in the trades,detail,price_days,first,last
0,AEVA,wrong company (section 5),disclosed company moved to DVAX,411,2025-01-10,2026-08-31
1,AMBC,renamed (section 5),all rows moved to OSG,218,2025-01-10,2025-11-20
2,ASGN,renamed (section 5),all rows moved to EFOR,323,2025-01-10,2026-04-24
3,ATGE,renamed (section 5),all rows moved to CVSA,280,2025-01-10,2026-02-23
4,AXL,renamed (section 5),all rows moved to DCH,268,2025-01-10,2026-02-04
5,BHLB,renamed (section 5),all rows moved to BBT,160,2025-01-10,2025-08-29
6,BK,renamed (section 5),all rows moved to BNY,342,2025-01-10,2026-05-21
7,BP,non-equity (section 5),rows dropped from the trades: BP CAPITAL MARKE...,411,2025-01-10,2026-08-31
8,CAST,non-equity (section 5),rows dropped from the trades: CAST CORP,121,2026-03-10,2026-08-31
9,CBFV,wrong company (section 5),disclosed company moved to PNC,411,2025-01-10,2026-08-31


### 1.1) What changed on the way, ticker by ticker

Three lists, all from the Step 2.5 review file, plus the five tickers left out of the purchase table. Together they are the complete account of how the price file's symbols relate to the final trades file.

In [3]:
rows_now = trades.groupby("Ticker").size()
pairs = wc[["Ticker", "Company", "CorrectedTicker", "CorrectedCompanyName", "RowsInTrades", "Reason"]].rename(columns={"Ticker": "old_ticker", "CorrectedTicker": "new_ticker"})
pairs["new_ticker_rows_now"] = pairs["new_ticker"].map(rows_now).fillna(0).astype(int)
print(f"wrong-company pairs: {len(pairs)} pairs, {pairs['RowsInTrades'].sum()} rows moved"); display(pairs)
renames = rn[["OldTicker", "LatestTicker", "Chain", "Source", "RowsOld", "Decision"]].copy()
renames["latest_rows_now"] = renames["LatestTicker"].map(rows_now).fillna(0).astype(int)
renames["old_rows_now"] = renames["OldTicker"].map(rows_now).fillna(0).astype(int)
print(f"symbol changes: {len(renames)} rows ({renames['Decision'].value_counts().to_dict()}); applied rows moved: {renames.loc[renames['Decision']=='apply','RowsOld'].sum()}"); display(renames)
dropped = pd.concat([
    ne.assign(what="non-equity, rows dropped from the trades (section 5)")[["what", "Ticker", "Company", "RowsInTrades", "Reason"]].rename(columns={"Ticker": "ticker", "Company": "company", "RowsInTrades": "rows", "Reason": "reason"}),
    left_out.groupby(["ticker", "Company", "reason"]).size().reset_index(name="rows").assign(what="purchases left out of step 5 (no usable FMP series)").rename(columns={"Company": "company"})[["what", "ticker", "company", "rows", "reason"]],
], ignore_index=True)
print(f"dropped: {len(dropped)} lines, {dropped['rows'].sum()} rows"); display(dropped)

wrong-company pairs: 27 pairs, 184 rows moved


,old_ticker,Company,new_ticker,CorrectedCompanyName,RowsInTrades,Reason,new_ticker_rows_now
0,NCL,NORTHERN TR CORP,NTRS,Northern Trust Corporation,6,Price fetch (Step 3): the ticker's price serie...,17
1,HSAI,HESS CORP,HES,Hess Corporation,6,Price fetch (Step 3): the ticker's price serie...,6
2,MMED,INARI MED INC,NARI,"Inari Medical, Inc.",5,Price fetch (Step 3): the ticker's price serie...,5
3,CNDT,CONFLUENT INC,CFLT,"Confluent, Inc.",2,Price fetch (Step 3): the ticker's price serie...,2
4,AEVA,DYNAVAX TECHNOLOGIES COR,DVAX,Dynavax Technologies Corporation,11,Price fetch (Step 3): the ticker's price serie...,11
5,BOX,JACK IN THE BOX INC,JACK,Jack in the Box Inc.,11,Price fetch (Step 3): the ticker's price serie...,11
6,CBFV,PNC FINANCIAL SERVICES GROUP,PNC,"The PNC Financial Services Group, Inc.",6,Price fetch (Step 3): the ticker's price serie...,45
7,CBFV,PNC FINANCIAL SERVICES GROUP I,PNC,"The PNC Financial Services Group, Inc.",2,Price fetch (Step 3): the ticker's price serie...,45
8,COFS,DISCOVER FINANCIAL SERVICES,DFS,Discover Financial Services,2,Price fetch (Step 3): the ticker's price serie...,7
9,HCA,ACADIA HEALTHCARE CO INC,ACHC,"Acadia Healthcare Company, Inc.",4,Price fetch (Step 3): the ticker's price serie...,4


symbol changes: 25 rows ({'apply': 21, 'check': 2, 'skip': 2}); applied rows moved: 201


,OldTicker,LatestTicker,Chain,Source,RowsOld,Decision,latest_rows_now,old_rows_now
0,AMBC,OSG,AMBC->OSG (2025-11-20),FMP symbol-change,3,apply,3,0
1,AROC,IDLDX,AROC->IDLDX (2025-05-27),FMP symbol-change,18,check,0,18
2,ASGN,EFOR,ASGN->EFOR (2026-04-24),FMP symbol-change,1,apply,3,0
3,ATGE,CVSA,ATGE->CVSA (2026-02-24),FMP symbol-change,10,apply,12,0
4,AXL,DCH,AXL->DCH (2026-02-05),FMP symbol-change,5,apply,7,0
5,BHLB,BBT,BHLB->BBT (2025-09-02),FMP symbol-change,6,apply,14,0
6,BK,BNY,BK->BNY (2026-05-21),FMP symbol-change,23,apply,36,0
7,CSWI,CSW,CSWI->CSW (2025-06-09),FMP symbol-change,24,apply,24,0
8,EQR,VMRK,EQR->VMRK (2026-08-18),FMP symbol-change,26,apply,26,0
9,HLX,HOS,HLX->HOS (2026-09-02),FMP symbol-change,12,skip,0,12


dropped: 10 lines, 42 rows


,what,ticker,company,rows,reason
0,"non-equity, rows dropped from the trades (sect...",BLK,BLACKROCK FDG INC,1,BlackRock Funding Inc. is BlackRock's debt-iss...
1,"non-equity, rows dropped from the trades (sect...",BLK,BLACKROCK FUNDING INC/DE,6,BlackRock Funding Inc. is BlackRock's debt-iss...
2,"non-equity, rows dropped from the trades (sect...",BP,BP CAPITAL MARKETS PLC,9,BP Capital Markets plc is BP's financing vehic...
3,"non-equity, rows dropped from the trades (sect...",SCGLY,SOCIETE GENERALE SA,4,Bought in the same weeks as the BP notes; trea...
4,"non-equity, rows dropped from the trades (sect...",CAST,CAST CORP,1,Issuer unidentifiable (FMP's CAST is FreeCast ...
5,purchases left out of step 5 (no usable FMP se...,BRKL,BROOKLINE BANCORP INC DE,4,Brookline Bancorp merged into Berkshire Hills ...
6,purchases left out of step 5 (no usable FMP se...,NVRI,ENVIRI CORP,6,Enviri re-registered in 2026; FMP's NVRI serie...
7,purchases left out of step 5 (no usable FMP se...,PINC,PREMIER INC,6,Premier Inc. was taken private in 2025; FMP's ...
8,purchases left out of step 5 (no usable FMP se...,RNAM,AVIDITY BIOSCIENCES INC,1,"Avidity Biosciences traded as RNA, a symbol no..."
9,purchases left out of step 5 (no usable FMP se...,WOLF,WOLFSPEED INC,4,Wolfspeed's old shares were cancelled in the C...


## 2) Price file integrity

Structural checks on the series step 5 reads: duplicate ticker-days, missing or non-positive prices, dates off the trading calendar, and holes inside a series. A hole followed by one or two prints that repeat the last price is the FMP pattern for a stock that stopped trading (merger closed) and received a few stale quotes afterwards; those prints give zero returns and should not be read as trading days.

In [4]:
Wp = W[P]
checks = {
    "duplicate (ticker, date) rows": int(prices.duplicated(["ticker", "date"]).sum()),
    "missing adj_close": int(prices["adj_close"].isna().sum()),
    "adj_close <= 0": int((prices["adj_close"] <= 0).sum()),
    "sample dates not on the French trading calendar": len(set(W.loc[SAMPLE_START:SAMPLE_END].index) - set(cal)),
    "calendar days with no price for any ticker": len(set(cal) - set(W.index)),
}
for k, v in checks.items(): print(f"{v:>6}  {k}")
first = Wp.apply(lambda s: s.first_valid_index()); last = Wp.apply(lambda s: s.last_valid_index())
last_buy = purch.groupby("ticker")["trade_date"].max()
holes = []
for t in P:
    s = Wp[t].dropna(); span = cal[(cal >= max(s.index.min(), cal.min())) & (cal <= s.index.max())]
    miss = span.difference(s.index)
    if len(miss):
        after = s.loc[miss.max():]
        holes.append((t, len(miss), miss.min().date(), miss.max().date(), len(after), round(float(s.loc[:miss.min()].iloc[-1]), 4), round(float(after.iloc[-1]), 4),
                      last_buy.get(t, pd.NaT).date() if t in last_buy else None, bool(((purch.loc[purch['ticker']==t, 'trade_date'] >= miss.min() - pd.Timedelta(days=20)).any()))))
holes = pd.DataFrame(holes, columns=["ticker", "missing_days", "hole_from", "hole_to", "prints_after_hole", "price_before_hole", "last_print", "last_purchase", "purchase_window_touches_hole"])
print(f"series with holes inside their span: {len(holes)}"); display(holes)
flat = {t: int(Wp[t].dropna().groupby((Wp[t].dropna().diff() != 0).cumsum()).size().max()) for t in P}
flat = pd.Series(flat); print(f"longest run of identical prices: {flat.max()} days ({flat.idxmax()}); runs >= 10 days: {(flat >= 10).sum()}")

     0  duplicate (ticker, date) rows
     0  missing adj_close
     0  adj_close <= 0
     0  sample dates not on the French trading calendar
     0  calendar days with no price for any ticker


series with holes inside their span: 5


,ticker,missing_days,hole_from,hole_to,prints_after_hole,price_before_hole,last_print,last_purchase,purchase_window_touches_hole
0,GDEN,5,2026-05-08,2026-05-14,1,28.55,28.55,2025-10-23,False
1,HBI,1,2025-12-01,2025-12-01,1,6.47,6.47,2025-09-22,False
2,LBRDK,1,2026-08-20,2026-08-20,1,36.02,36.02,2026-06-03,False
3,MCW,10,2026-05-22,2026-06-05,1,7.10,7.10,2025-11-24,False
4,VTLE,4,2025-12-15,2025-12-18,1,17.92,17.92,2025-05-29,False


longest run of identical prices: 7 days (CWAN); runs >= 10 days: 0


## 3) Where each series starts and ends, against the entity table

A series that ends before 31 August 2026 should belong to a company with a delisting or merger event (Step 4.2's entity table), and the last price should sit a few days before the event date (FMP dates the event at deregistration). A series that starts after 10 January 2025 should belong to a company listed during the sample, with its first purchase after the first price. Anything else would mean the symbol carries another issuer's history.

In [5]:
ev = ent[ent["ticker"].isin(P) & ent["event_type"].notna()].groupby("ticker").agg(events=("event_type", lambda s: ",".join(sorted(set(s)))), event_date=("event_date", "max"))
ev["event_date"] = pd.to_datetime(ev["event_date"])
def incomplete_windows(t, end):
    d = purch.loc[purch["ticker"] == t, "trade_date"]; i = cal.searchsorted(d.values)
    return int(sum(1 for k in i if k + EVENT_DAYS >= len(cal) or cal[min(k + EVENT_DAYS, len(cal) - 1)] > end))
ends = pd.DataFrame({"last_price": last[last < SAMPLE_END]}).join(ev)
ends["last_purchase"] = last_buy.reindex(ends.index)
ends["days_price_to_event"] = (ends["event_date"] - ends["last_price"]).dt.days
ends["purchases_with_cut_window"] = [incomplete_windows(t, ends.loc[t, "last_price"]) for t in ends.index]
ends = ends.sort_values("last_price")
print(f"series ending before {SAMPLE_END.date()}: {len(ends)} | with a delisting/merger event: {ends['events'].str.contains('delisted|merged', na=False).sum()} | without event: {ends['events'].isna().sum()}")
print(f"event date minus last price, days: median {ends['days_price_to_event'].median():.0f}, max {ends['days_price_to_event'].max():.0f}; series ending AFTER the event date: {(ends['days_price_to_event'] < 0).sum()}")
print(f"purchases whose 10-day window is cut by the series end: {ends['purchases_with_cut_window'].sum()} on {(ends['purchases_with_cut_window'] > 0).sum()} tickers")
display(ends)
first_buy = purch.groupby("ticker")["trade_date"].min()
starts = pd.DataFrame({"first_price": first[first > W.index.min()]}).join(ev); starts["first_purchase"] = first_buy.reindex(starts.index)
starts["purchase_before_first_price"] = starts["first_purchase"] < starts["first_price"]
print(f"series starting after {W.index.min().date()}: {len(starts)} | listed during the sample per entity table: {starts['events'].str.contains('listed_during_sample', na=False).sum()} (the rest: {', '.join(starts.index[~starts['events'].str.contains('listed_during_sample', na=False)])}, a 2025 spin-off the entity table did not date) | purchases before the first price: {int(starts['purchase_before_first_price'].sum())}")
display(starts.sort_values("first_price"))

# EndsEarly: why each series ends (entity table); StartsLate: beta method from step 2. A ReviewNote written in either sheet survives a rerun.
def carry_note(df, sheet):
    try:
        prev = pd.read_excel(OUT_XLSX, sheet_name=sheet).set_index("ticker")
    except ValueError:
        return ""
    return df.index.map(prev["ReviewNote"]).fillna("") if "ReviewNote" in prev else ""
ends["reading"] = [f"{r.events} on {r.event_date.date()} (entity table); last price {int(r.days_price_to_event)} days before" if pd.notna(r.event_date)
                   else (f"{r.events}: no deregistration date in the entity table" if isinstance(r.events, str) else "no event in the entity table")
                   for r in ends.itertuples()]
ends["ReviewNote"] = carry_note(ends, "EndsEarly")
beta_method = pd.read_csv("../2) ComputeReturns/betas_presample.csv").query("model == 'capm'").set_index("ticker")["method"]
starts["beta_method"] = starts.index.map(beta_method)
starts["ReviewNote"] = carry_note(starts, "StartsLate")

series ending before 2026-08-31: 76 | with a delisting/merger event: 76 | without event: 0
event date minus last price, days: median 11, max 23; series ending AFTER the event date: 3
purchases whose 10-day window is cut by the series end: 13 on 9 tickers


,last_price,events,event_date,last_purchase,days_price_to_event,purchases_with_cut_window
ticker,,,,,,
ROIC,2025-02-12,delisted,2025-02-24,2025-01-31,12.0,2
NARI,2025-02-19,delisted,2025-03-03,2025-02-13,12.0,2
HCP,2025-02-27,delisted,2025-03-10,2025-01-30,11.0,0
AZPN,2025-03-12,delisted,2025-03-24,2025-03-05,12.0,1
SWI,2025-04-15,delisted,2025-04-28,2025-04-08,13.0,1
PDCO,2025-04-21,delisted,2025-04-28,2025-04-10,7.0,1
BERY,2025-04-29,delisted,2025-05-12,2025-03-05,13.0,0
DFS,2025-05-16,delisted,2025-05-29,2025-05-15,13.0,2
ESGR,2025-07-01,delisted,2025-07-24,2025-06-05,23.0,0


series starting after 2025-01-10: 17 | listed during the sample per entity table: 16 (the rest: SNDK, a 2025 spin-off the entity table did not date) | purchases before the first price: 0


,first_price,events,event_date,first_purchase,purchase_before_first_price
ticker,,,,,
MRP,2025-02-05,listed_during_sample,2025-02-05,2025-02-13,False
SNDK,2025-02-13,unchanged,NaT,2025-03-11,False
SAIL,2025-02-13,"listed_during_sample,renamed",2025-02-13,2026-02-12,False
KRMN,2025-02-13,"listed_during_sample,renamed",2025-02-13,2026-07-17,False
CRWV,2025-03-28,listed_during_sample,2025-03-28,2025-09-16,False
AMRZ,2025-06-23,listed_during_sample,2025-06-23,2025-10-09,False
RAL,2025-06-25,listed_during_sample,2025-06-25,2025-07-23,False
FIG,2025-07-31,listed_during_sample,2025-07-31,2026-02-12,False
SOLS,2025-10-20,"listed_during_sample,renamed",2025-10-20,2025-12-22,False


## 4) Moves that look wrong

Four screens. (a) Every day with a move above 50 percent in absolute value, matched against the co-author's hand review (sheet `ExtremeMoves` of this workbook): every such day must have a decision. (b) Near-exact halvings or doublings, the signature of an unadjusted split; the flagged days are checked against FMP's full daily bar (open, high, low, volume) and split history, which costs one call per ticker. (c) Continuity across each symbol change Step 2.5 applied: a break on the rename date would mean two different series were glued together. (d) Moves between 20 and 50 percent that nobody has reviewed, listed as candidates only.

In [6]:
r = Wp.pct_change(fill_method=None).loc[SAMPLE_START:SAMPLE_END]
big = r.stack().rename("ret").reset_index(); big.columns = ["date", "ticker", "ret"]
big = big[big["ret"].abs() > 0.5].merge(bad_rev[["Ticker", "Date", "Decision", "Comment"]].rename(columns={"Ticker": "ticker", "Date": "date"}), on=["ticker", "date"], how="left").sort_values("date")
big["prev_close"] = [Wp[t].loc[:d].dropna().iloc[-2] for t, d in zip(big["ticker"], big["date"])]
big["close"] = [Wp.at[d, t] for t, d in zip(big["ticker"], big["date"])]
big["purchases_20d_before"] = [int(((purch["ticker"] == t) & purch["trade_date"].between(d - pd.Timedelta(days=20), d)).sum()) for t, d in zip(big["ticker"], big["date"])]
print(f"(a) days with |ret| > 50%: {len(big)}; reviewed by Filip: {big['Decision'].notna().sum()}; decisions: {big['Decision'].value_counts().to_dict()}; unreviewed: {int(big['Decision'].isna().sum())}")
display(big[["date", "ticker", "prev_close", "close", "ret", "Decision", "Comment", "purchases_20d_before"]])
ratio = (1 + r).stack()
targets = {0.5: 0.01, 2: 0.03, 1/3: 0.01, 3: 0.05, 0.25: 0.01, 4: 0.05, 0.1: 0.005, 10: 0.1}
near = ratio[np.logical_or.reduce([ratio.sub(k).abs() < tol for k, tol in targets.items()])].rename("ratio").reset_index(); near.columns = ["date", "ticker", "ratio"]
print(f"(b) near-exact split ratios: {len(near)}"); display(near)

(a) days with |ret| > 50%: 25; reviewed by Filip: 25; decisions: {'genuine': 23, 'wrong': 2}; unreviewed: 0


,date,ticker,prev_close,close,ret,Decision,Comment,purchases_20d_before
0,2025-03-31,CORT,54.63,114.22,1.090793,genuine,Phase 3 cancer result here too,0
1,2025-04-16,HTZ,3.65,5.71,0.564384,genuine,Ackman / Pershing Square took a stake,0
2,2025-05-15,FL,12.87,23.90,0.857032,genuine,Takeover announced,0
3,2025-05-22,AAP,30.38,47.72,0.570770,genuine,"Beat estimates, kept full-year guidance",0
4,2025-06-23,SPTN,17.49,26.35,0.506575,genuine,"Takeover announced, $26.90 cash. Not on Yahoo",0
5,2025-07-10,MP,30.03,45.23,0.506161,genuine,Deal with the US Department of Defense,0
6,2025-07-22,MEDP,308.88,477.73,0.546652,genuine,"Beat estimates, raised full-year guidance",0
7,2025-08-26,ECHO,29.88,50.87,0.702477,genuine,AT&T buys spectrum for $23bn,0
8,2025-10-29,LRN,153.53,70.05,-0.543737,genuine,"Beat Q1, but weak FY26 guidance",0
9,2025-10-29,PUMP,7.30,11.10,0.520548,genuine,Q3 above estimates,0


(b) near-exact split ratios: 3


,date,ticker,ratio
0,2025-12-31,CORT,0.495797
1,2026-05-06,PRIM,0.498864
2,2026-08-10,ACH,0.500000


In [7]:
# (b) continued: the flagged days against FMP's full daily bar and split history (one profile-free call per ticker and day; skipped without a key)
env = Path("../../.env"); KEY = next((l.split("=", 1)[1].strip().strip('"') for l in env.read_text().splitlines() if l.startswith("FMP_API_KEY")), "") if env.exists() else ""
def fmp(ep, **p):
    try:
        x = requests.get(f"https://financialmodelingprep.com/stable/{ep}", params={**p, "apikey": KEY}, timeout=30); return x.json() if x.ok else None
    except requests.RequestException:
        return None
verify = []
if KEY:
    for t, d in zip(near["ticker"], near["date"]):
        bars = fmp("historical-price-eod/full", symbol=t, **{"from": (d - pd.Timedelta(days=5)).date().isoformat(), "to": (d + pd.Timedelta(days=3)).date().isoformat()}) or []
        bars = sorted(bars, key=lambda b: b["date"]); splits = fmp("splits", symbol=t) or []
        day = next((b for b in bars if b["date"] == d.date().isoformat()), None); prev = next((b for b in reversed(bars) if b["date"] < d.date().isoformat()), None)
        med_vol = np.median([b["volume"] for b in bars if b["date"] != d.date().isoformat()]) if len(bars) > 1 else np.nan
        split_near = [s["date"] for s in splits if abs((pd.Timestamp(s["date"]) - d).days) <= 5]
        if day and prev:
            verify.append((t, d.date(), prev["close"], day["open"], day["low"], day["high"], day["close"], day["volume"], round(day["volume"] / med_vol, 1) if med_vol else np.nan,
                           round(day["close"] / prev["close"] - 1, 3), round(day["open"] / prev["close"] - 1, 3), split_near or "none",
                           "genuine: opened far below the previous close, traded through the day on many times the normal volume, no split" if day["open"] / prev["close"] < 0.8 and (day["volume"] / med_vol if med_vol else 0) > 3 and not split_near else "check"))
verify = pd.DataFrame(verify, columns=["ticker", "date", "prev_close", "open", "low", "high", "close", "volume", "volume_x_normal", "close_to_close", "open_vs_prev_close", "split_within_5d", "reading"])
print("FMP check of the near-exact ratios:" if KEY else "no FMP key found; the near-exact ratios are not verified here"); display(verify)

FMP check of the near-exact ratios:


,ticker,date,prev_close,open,low,high,close,volume,volume_x_normal,close_to_close,open_vs_prev_close,split_within_5d,reading
0,CORT,2025-12-31,70.19,38.89,32.99,39.00,34.80,20305000,12.3,-0.504,-0.446,none,"genuine: opened far below the previous close, ..."
1,PRIM,2026-05-06,202.92,128.39,98.09,133.85,101.23,13165521,5.5,-0.501,-0.367,none,"genuine: opened far below the previous close, ..."
2,ACH,2026-08-10,2.80,1.74,1.24,1.74,1.40,8094900,4.9,-0.500,-0.379,none,"genuine: opened far below the previous close, ..."


In [8]:
# (c) continuity at the symbol changes Step 2.5 applied: the adjusted close on the days around the rename date
cont = []
for _, x in apply.iterrows():
    m = re.search(r"\((\d{4}-\d{2}-\d{2})\)", str(x["Chain"])); t = x["LatestTicker"]
    if t not in W.columns: cont.append((x["OldTicker"], t, None, "no series", np.nan)); continue
    if not m: cont.append((x["OldTicker"], t, None, "when-issued symbol, no change date (hand review)", np.nan)); continue
    d = pd.Timestamp(m.group(1)); s = W[t].dropna()
    if not (s.index.min() <= d <= s.index.max()): cont.append((x["OldTicker"], t, d.date(), f"change date outside the series ({s.index.min().date()} .. {s.index.max().date()}): series already under the new symbol", np.nan)); continue
    seg = s.loc[d - pd.Timedelta(days=4): d + pd.Timedelta(days=4)]
    cont.append((x["OldTicker"], t, d.date(), "continuous" if seg.pct_change().abs().max() < 0.15 else "large move around the change date (see section 4a)", round(float(seg.pct_change().abs().max()), 3)))
cont = pd.DataFrame(cont, columns=["old_ticker", "latest_ticker", "change_date", "series_at_change", "max_abs_daily_move_4d"])
print(f"(c) symbol changes applied: {len(cont)}; continuous or already under the new symbol: {cont['series_at_change'].str.startswith(('continuous', 'change date outside', 'when-issued')).sum()}"); display(cont)
# (d) unreviewed moves between 20% and 50%
mid = r.stack().rename("ret").reset_index(); mid.columns = ["date", "ticker", "ret"]
mid = mid[(mid["ret"].abs() > 0.2) & (mid["ret"].abs() <= 0.5)]
mid["purchases_20d_before"] = [int(((purch["ticker"] == t) & purch["trade_date"].between(d - pd.Timedelta(days=20), d)).sum()) for t, d in zip(mid["ticker"], mid["date"])]
print(f"(d) days with 20% < |ret| <= 50%: {len(mid)} on {mid['ticker'].nunique()} tickers; of these inside a purchase window (purchase in the 20 days before): {(mid['purchases_20d_before'] > 0).sum()}")
display(mid.sort_values("ret").head(15)); display(mid.sort_values("ret").tail(10))

(c) symbol changes applied: 21; continuous or already under the new symbol: 19


,old_ticker,latest_ticker,change_date,series_at_change,max_abs_daily_move_4d
0,AMBC,OSG,2025-11-20,continuous,0.038
1,ASGN,EFOR,2026-04-24,large move around the change date (see section...,0.517
2,ATGE,CVSA,2026-02-24,continuous,0.035
3,AXL,DCH,2026-02-05,continuous,0.095
4,BHLB,BBT,2025-09-02,continuous,0.013
5,BK,BNY,2026-05-21,continuous,0.013
6,CSWI,CSW,2025-06-09,continuous,0.028
7,EQR,VMRK,2026-08-18,continuous,0.035
8,HONAV,HONA,None,"when-issued symbol, no change date (hand review)",NaN
9,IAC,PPLI,2026-06-04,continuous,0.056


(d) days with 20% < |ret| <= 50%: 818 on 457 tickers; of these inside a purchase window (purchase in the 20 days before): 176


,date,ticker,ret,purchases_20d_before
620084,2026-08-10,ACH,-0.500000,0
312323,2025-10-29,VRNS,-0.486667,0
118609,2025-05-07,HAIN,-0.476534,0
611476,2026-07-31,MYGN,-0.467412,0
312991,2025-10-30,FMC,-0.465182,0
441323,2026-02-26,THRY,-0.464286,0
77603,2025-03-31,PCVX,-0.456378,0
601774,2026-07-23,LESL,-0.448133,0
311386,2025-10-29,FISV,-0.440438,2
520236,2026-05-08,FWRD,-0.430467,0


,date,ticker,ret,purchases_20d_before
27243,2025-02-13,AUR,0.460465,0
620779,2026-08-10,HZO,0.460762,0
216518,2025-08-05,STAA,0.461330,0
217758,2025-08-06,MYGN,0.465116,1
546654,2026-06-02,VSXY,0.474401,0
446314,2026-03-03,ZD,0.480900,0
66496,2025-03-20,PRA,0.481338,0
427550,2026-02-13,CRSR,0.482533,0
161457,2025-06-16,ECHO,0.491093,0
254427,2025-09-09,NBIS,0.494224,0


### 4.1) Dividend adjustment: the one series where it fails

FMP's dividend-adjusted close should show no drop on an ex-dividend date. The screen above found one company whose adjusted series sits near zero and still falls by a quarter and a third on two days: Elme Communities (ELME), which sold its portfolio in 2025–26 and paid out the proceeds. FMP's dividend table has the 8 January 2026 distribution (adjusted away) but not the two later ones, so the drops on 26 January and 25 June 2026 are payouts a shareholder received, not losses. The cell checks the raw bars and the dividend table for ELME (two calls) and records the finding; whether other liquidations exist in the sample is caught only to the extent they produce a move above 20 percent (section 4d).

In [9]:
elme = pd.DataFrame()
if "ELME" in W.columns:
    e_r = W["ELME"].pct_change(); drops = e_r[e_r.abs() > 0.2]
    divs = fmp("dividends", symbol="ELME") if KEY else None
    div_dates = sorted(d["date"] for d in (divs or []) if d["date"] >= "2025-01-01")
    rows = []
    for d, v in drops.items():
        bars = fmp("historical-price-eod/full", symbol="ELME", **{"from": (d - pd.Timedelta(days=4)).date().isoformat(), "to": d.date().isoformat()}) if KEY else None
        bars = sorted(bars or [], key=lambda b: b["date"]); day = bars[-1] if bars and bars[-1]["date"] == d.date().isoformat() else None; prev = bars[-2] if len(bars) > 1 else None
        rows.append((d.date(), round(float(v), 3), round(float(W.at[d, "ELME"]), 4), prev["close"] if prev else np.nan, day["close"] if day else np.nan, day["volume"] if day else np.nan,
                     d.date().isoformat() in div_dates, "no ex-dividend date in FMP's table: an unadjusted liquidating distribution; the return is not a loss" if d.date().isoformat() not in div_dates else "ex-dividend date known: adjustment missing"))
    elme = pd.DataFrame(rows, columns=["date", "adj_return", "adj_close", "raw_prev_close", "raw_close", "volume", "in_fmp_dividend_table", "reading"])
    print("ELME adjusted close over the sample:", round(float(W['ELME'].loc[SAMPLE_START]), 4), "->", round(float(W['ELME'].loc[SAMPLE_END]), 4), "| FMP dividends since 2025:", [(d["date"], d["dividend"]) for d in (divs or []) if d["date"] >= "2025-01-01"])
    print("ELME purchases:", purch.loc[purch['ticker']=='ELME', 'trade_date'].dt.date.tolist())
    display(elme)

ELME adjusted close over the sample: 0.0374 -> 0.0285 | FMP dividends since 2025: [('2026-10-23', 1.74), ('2026-01-08', 14.67), ('2025-09-17', 0.18), ('2025-06-17', 0.18), ('2025-03-19', 0.18)]
ELME purchases: [datetime.date(2025, 1, 30), datetime.date(2025, 2, 3), datetime.date(2025, 2, 5), datetime.date(2025, 4, 8), datetime.date(2025, 4, 10)]


,date,adj_return,adj_close,raw_prev_close,raw_close,volume,in_fmp_dividend_table,reading
0,2026-01-26,-0.247,0.0378,2.96,2.23,23098400,False,no ex-dividend date in FMP's table: an unadjus...
1,2026-06-25,-0.332,0.0229,2.02,1.35,16979957,False,no ex-dividend date in FMP's table: an unadjus...


## 5) Summary and export

One sheet per table above. The three findings that need a decision are repeated in the summary.

In [10]:
decisions = [
    ("stale prints after a delisting", f"{len(holes)} series ({', '.join(holes['ticker'])}) have a hole followed by repeated last prices; zero returns on non-trading days. No purchase window touches them. Suggested: blank prices after the hole in step 5, or leave as is (no effect on the results)."),
    ("series cut by a takeover", f"{ends['purchases_with_cut_window'].sum()} purchases on {(ends['purchases_with_cut_window'] > 0).sum()} tickers have fewer than 10 trading days after the purchase because the company was taken over. Rule needed: count the available days or drop the purchase."),
    ("ELME liquidating distributions", f"{len(elme)} days with an unadjusted payout ({', '.join(str(x) for x in elme['date'])}); returns of {', '.join(f'{v:.0%}' for v in elme['adj_return'])} are not losses. No ELME purchase window touches them. Suggested: blank the two days (mark them wrong in the sheet ExtremeMoves) or drop ELME." if len(elme) else "not checked (no key)"),
    ("moves above 50%", f"{len(big)} days, all in Filip's review: {big['Decision'].value_counts().to_dict()}; the near-exact halvings ({', '.join(near['ticker'])}) are genuine on FMP's full bars (no split, opened far below the previous close, many times the normal volume)." if len(verify) and (verify['reading'].str.startswith('genuine')).all() else f"{len(big)} days; see table 4a/4b"),
]
summary = pd.DataFrame([
    ("trades tickers / purchased / sale-only", f"{len(T_all)} / {len(T_buy)} / {len(sale_only)}"),
    ("step-5 tickers / purchases", f"{len(P)} / {len(purch)}"),
    ("purchases left out (no usable FMP series)", f"{len(left_out)} on {left_out['ticker'].nunique()} tickers"),
    ("price symbols / used / superseded", f"{len(PR)} / {len(PR & set(P))} / {len(PR - T_all)}"),
    ("wrong-company pairs / rows moved (section 5)", f"{len(pairs)} / {pairs['RowsInTrades'].sum()}"),
    ("symbol changes applied / rows moved (section 5)", f"{(renames['Decision']=='apply').sum()} / {renames.loc[renames['Decision']=='apply','RowsOld'].sum()}"),
    ("integrity: duplicates / missing / non-positive / off-calendar", " / ".join(str(v) for v in checks.values())),
    ("series with holes (stale prints)", len(holes)),
    ("series ending early / all with a delisting event", f"{len(ends)} / {ends['events'].str.contains('delisted|merged', na=False).all()}"),
    ("series starting late / all listed during the sample", f"{len(starts)} / {starts['events'].str.contains('listed_during_sample', na=False).all()}"),
    ("purchases with a window cut by a takeover", int(ends['purchases_with_cut_window'].sum())),
    ("days |ret| > 50% / reviewed / removed by step 5", f"{len(big)} / {big['Decision'].notna().sum()} / {(big['Decision']=='wrong').sum()}"),
    ("near-exact split ratios / verified genuine", f"{len(near)} / {(verify['reading'].str.startswith('genuine')).sum() if len(verify) else 'n/a'}"),
    ("symbol changes with a continuous series", f"{cont['series_at_change'].str.startswith(('continuous', 'change date outside', 'when-issued')).sum()} of {len(cont)}"),
    ("unreviewed days 20% < |ret| <= 50% / inside a purchase window", f"{len(mid)} / {(mid['purchases_20d_before'] > 0).sum()}"),
    ("ELME unadjusted payout days", len(elme)),
] + [("DECISION: " + k, v) for k, v in decisions], columns=["item", "value"])
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl", mode="a", if_sheet_exists="replace") as w:   # keeps the hand-edited ExtremeMoves sheets
    summary.to_excel(w, sheet_name="Summary", index=False); universe.to_excel(w, sheet_name="Universe", index=False)
    superseded.to_excel(w, sheet_name="SupersededSymbols", index=False); pairs.to_excel(w, sheet_name="WrongCompanyPairs", index=False)
    renames.to_excel(w, sheet_name="SymbolChanges", index=False); dropped.to_excel(w, sheet_name="Dropped", index=False)
    holes.to_excel(w, sheet_name="StalePrints", index=False)
    ends.reset_index().rename(columns={"index": "ticker"}).assign(last_price=lambda d: d["last_price"].dt.date, event_date=lambda d: d["event_date"].dt.date, last_purchase=lambda d: d["last_purchase"].dt.date).to_excel(w, sheet_name="EndsEarly", index=False)
    starts.reset_index().rename(columns={"index": "ticker"}).assign(first_price=lambda d: d["first_price"].dt.date, first_purchase=lambda d: d["first_purchase"].dt.date).to_excel(w, sheet_name="StartsLate", index=False)
    big.assign(date=big["date"].dt.date).to_excel(w, sheet_name="BigMoves", index=False); verify.to_excel(w, sheet_name="SplitCheck", index=False)
    cont.to_excel(w, sheet_name="RenameContinuity", index=False); mid.assign(date=mid["date"].dt.date).sort_values("ret").to_excel(w, sheet_name="Unreviewed20to50", index=False)
    elme.to_excel(w, sheet_name="ELME", index=False)
display(summary)
print(f"written: {OUT_XLSX} ({len(pd.ExcelFile(OUT_XLSX).sheet_names)} sheets)")

,item,value
0,trades tickers / purchased / sale-only,1620 / 1599 / 21
1,step-5 tickers / purchases,1594 / 18797
2,purchases left out (no usable FMP series),21 on 5 tickers
3,price symbols / used / superseded,1639 / 1594 / 40
4,wrong-company pairs / rows moved (section 5),27 / 184
5,symbol changes applied / rows moved (section 5),21 / 201
6,integrity: duplicates / missing / non-positive...,0 / 0 / 0 / 0 / 0
7,series with holes (stale prints),5
8,series ending early / all with a delisting event,76 / True
9,series starting late / all listed during the s...,17 / False


written: PricesReview.xlsx (17 sheets)


→ The price file covers every purchased ticker the trades file has, the superseded symbols are explained by the Step 2.5 review, and the series behave as the entity table says they should. What remains are three decisions for the authors (takeover-cut windows, ELME's payouts, stale prints), none of which touches a purchase window except the takeover rule.